In [1]:
!pip install -q transformers datasets sentencepiece psutil scikit-learn accelerate bitsandbytes
print("Done!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00
Done!


In [5]:
from google.colab import files
uploaded = files.upload()  # upload baseline_model.zip AND results.zip

!unzip -q baseline_model.zip -d /content/
!unzip -q results.zip -d /content/
print("Files extracted!")
!ls /content/content/baseline_model/
!ls /content/content/results/

Saving baseline_model.zip to baseline_model (1).zip
Saving results.zip to results (2).zip
replace /content/content/baseline_model/tokenizer.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/content/baseline_model/tokenizer_config.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/content/baseline_model/config.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/content/baseline_model/model.safetensors? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/content/results/train.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/content/results/all_results.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/content/results/test.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
Files extracted!
config.json  model.safetensors	tokenizer_config.json  tokenizer.json
all_results.json  test.csv  train.csv


In [6]:
import os, time, json, gc
import numpy as np
import torch
import psutil
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

MODEL_DIR  = '/content/content/baseline_model' # Corrected path
MAX_LENGTH = 128
BATCH_SIZE = 16
NUM_LABELS = 7
label2id   = {'neutral':0,'joy':1,'anger':2,'surprise':3,'sadness':4,'fear':5,'disgust':6}
id2label   = {v: k for k, v in label2id.items()}

with open('/content/content/results/all_results.json') as f: # Corrected path
    ALL = json.load(f)
print("Baseline results loaded:", ALL)

Device: cpu
Baseline results loaded: {'params': 33448967, 'baseline_f1': 0.3064, 'baseline_size_mb': 141.9, 'baseline_latency_ms': 9.23, 'baseline_ram_mb': 0.0}


In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, keep_accents=True)
test_df   = pd.read_csv('/content/content/results/test.csv') # Corrected path

class HindiDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.texts  = df['Sentence'].tolist()
        self.labels = df['label'].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=MAX_LENGTH,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

test_dataset = HindiDataset(test_df)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)
print("Test samples:", len(test_dataset))

Test samples: 1600


In [9]:
def benchmark(model, name, save_path=None):
    m_device = next(model.parameters()).device
    model.eval()

    # F1
    preds_all, labs_all = [], []
    with torch.no_grad():
        for batch in test_loader:
            labels = batch.pop('labels').to(m_device)
            batch  = {k: v.to(m_device) for k, v in batch.items()}
            preds  = model(**batch).logits.argmax(dim=-1).cpu().numpy()
            preds_all.extend(preds)
            labs_all.extend(labels.cpu().numpy())
    f1 = f1_score(labs_all, preds_all, average='macro')

    # Size
    if save_path:
        torch.save(model.state_dict(), save_path)
        size_mb = os.path.getsize(save_path) / 1024**2
    else:
        size_mb = sum(p.numel() for p in model.parameters()) * 4 / 1024**2

    # Latency
    sample = tokenizer(
        "यह बहुत अच्छा है।", return_tensors='pt',
        truncation=True, max_length=MAX_LENGTH, padding='max_length'
    ).to(m_device)
    for _ in range(10):
        with torch.no_grad(): _ = model(**sample)
    times = []
    for _ in range(100):
        t = time.perf_counter()
        with torch.no_grad(): _ = model(**sample)
        times.append((time.perf_counter() - t) * 1000)

    # RAM
    proc  = psutil.Process(os.getpid())
    r0    = proc.memory_info().rss / 1024**2
    with torch.no_grad(): _ = model(**sample)
    ram   = proc.memory_info().rss / 1024**2 - r0

    result = {
        'f1':         round(f1, 4),
        'size_mb':    round(size_mb, 1),
        'latency_ms': round(np.mean(times), 2),
        'ram_mb':     round(ram, 1)
    }
    print(f"\n[{name}]", result)
    return result

In [11]:
# Must run on CPU
model_fp32 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR, num_labels=NUM_LABELS
)  # CPU by default — do NOT .to(device)

model_int8 = torch.quantization.quantize_dynamic(
    model_fp32, {torch.nn.Linear}, dtype=torch.qint8
)
print("INT8 applied!")

result_int8 = benchmark(model_int8, 'INT8', '/content/content/results/model_int8.pt') # Corrected save path
ALL['int8'] = result_int8

del model_int8, model_fp32
gc.collect()

Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

/tmp/ipykernel_25587/2847094768.py:6: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = torch.quantization.quantize_dynamic(


INT8 applied!

[INT8] {'f1': 0.0907, 'size_mb': 105.4, 'latency_ms': np.float64(331.22), 'ram_mb': 0.0}


669

In [12]:
from transformers import BitsAndBytesConfig

if not torch.cuda.is_available():
    print("No GPU — skipping INT4")
    ALL['int4'] = {'f1':'N/A','size_mb':'N/A','latency_ms':'N/A','ram_mb':'N/A'}
else:
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    model_int4 = AutoModelForSequenceClassification.from_pretrained(
        MODEL_DIR, num_labels=NUM_LABELS,
        quantization_config=bnb,
        device_map='auto'
    )
    result_int4 = benchmark(model_int4, 'INT4', '/content/results/model_int4.pt')
    ALL['int4'] = result_int4
    del model_int4
    gc.collect()
    torch.cuda.empty_cache()

No GPU — skipping INT4


In [20]:
# Clean up numpy types before saving
import json

def clean(obj):
    if isinstance(obj, dict):
        return {k: clean(v) for k, v in obj.items()}
    if hasattr(obj, 'item'):  # numpy scalar
        return obj.item()
    return obj

ALL = clean(ALL)

with open('/content/content/results/all_results.json', 'w') as f:
    json.dump(ALL, f, indent=2)

print("Cleaned results:")
print(json.dumps(ALL, indent=2))

!zip -r /content/results.zip /content/results
from google.colab import files
files.download('/content/results.zip')
print("✅ Notebook 2 done! Download the zip.")

Cleaned results:
{
  "params": 33448967,
  "baseline_f1": 0.3064,
  "baseline_size_mb": 141.9,
  "baseline_latency_ms": 9.23,
  "baseline_ram_mb": 0.0,
  "int8": {
    "f1": 0.0907,
    "size_mb": 105.4,
    "latency_ms": 331.22,
    "ram_mb": 0.0
  },
  "int4": {
    "f1": "N/A",
    "size_mb": "N/A",
    "latency_ms": "N/A",
    "ram_mb": "N/A"
  }
}
	zip warning: name not matched: /content/results

zip error: Nothing to do! (try: zip -r /content/results.zip . -i /content/results)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Notebook 2 done! Download the zip.
